<a href="https://colab.research.google.com/github/Aditya-V-Naidoo/UoY_NLP_practical/blob/main/NLPG_Practical_5_POS_tagging_HMM_Brill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical 5 — Part-of-speech tagging with HMM and Brill taggers

**NLPG: Natural Language Processing · University of York**

This practical asks what it means to assign a part-of-speech (PoS) tag to each token in a sentence. We shall examine two deliberately transparent approaches:

- a **Hidden Markov Model (HMM)** tagger, which chooses a probable sequence of hidden tags;
- a **Brill tagger**, which starts with a simple baseline and learns explicit correction rules.

We begin with a tiny example that can be inspected by hand, evaluate both approaches on held-out Wall Street Journal (WSJ) material, and finally apply them to a complete book.

There is nothing to submit. Keep brief notes on your predictions, observations and explanations for the summing-up lecture.

## Learning outcomes

By the end of the practical, you should be able to:

1. explain PoS tagging as **sequence labelling**, not merely dictionary lookup;
2. identify the observations and hidden states in an HMM tagger;
3. explain the roles of transition and emission probabilities;
4. explain how a Brill tagger improves a baseline using learned transformation rules;
5. evaluate taggers on held-out labelled data;
6. inspect disagreements and errors rather than relying only on overall accuracy;
7. distinguish **evaluation on labelled WSJ text** from **application to an unlabelled book**;
8. compare where the two approaches place their generalisations.

The working discipline throughout is:

> **Predict → run → observe → explain.**

## 1. Setup

The notebook uses NLTK's copy of the Penn Treebank WSJ corpus and its Gutenberg corpus. The first run in Colab may take a short time while these resources are downloaded.

In [1]:
import math
import time
from collections import Counter

import nltk
import pandas as pd
from IPython.display import display

for resource, location in [
    ("treebank", "corpora/treebank.zip"),
    ("gutenberg", "corpora/gutenberg.zip"),
    ("punkt", "tokenizers/punkt.zip"),
    ("punkt_tab", "tokenizers/punkt_tab.zip"),
]:
    try:
        nltk.data.find(location)
    except LookupError:
        nltk.download(resource, quiet=True)

from nltk.corpus import gutenberg, treebank
from nltk.probability import LidstoneProbDist
from nltk.tag import DefaultTagger, RegexpTagger, UnigramTagger, brill
from nltk.tag.brill_trainer import BrillTaggerTrainer
from nltk.tag.hmm import HiddenMarkovModelTrainer

print("NLTK version:", nltk.__version__)

NLTK version: 3.9.1


### A small Penn Treebank tag reminder

You have already encountered the WSJ tag set. These are some of the tags used below.

In [2]:
tag_reminder = pd.DataFrame(
    [
        ("DT", "determiner", "the, a"),
        ("NN", "singular/common noun", "fish, book"),
        ("NNS", "plural noun", "people, books"),
        ("PRP", "personal pronoun", "they, we"),
        ("MD", "modal", "can, might"),
        ("VB", "base-form verb", "fish, walk"),
        ("VBD", "past-tense verb", "escaped, walked"),
        ("VBG", "gerund/present participle", "fishing, walking"),
        ("VBP", "non-third-person present verb", "swim, walk"),
        ("VBZ", "third-person singular present verb", "swims, walks"),
        ("JJ", "adjective", "small, financial"),
        ("RB", "adverb", "quickly, not"),
    ],
    columns=["Tag", "Meaning", "Examples"],
)
display(tag_reminder)

,Tag,Meaning,Examples
0,DT,determiner,"the, a"
1,NN,singular/common noun,"fish, book"
2,NNS,plural noun,"people, books"
3,PRP,personal pronoun,"they, we"
4,MD,modal,"can, might"
5,VB,base-form verb,"fish, walk"
6,VBD,past-tense verb,"escaped, walked"
7,VBG,gerund/present participle,"fishing, walking"
8,VBP,non-third-person present verb,"swim, walk"
9,VBZ,third-person singular present verb,"swims, walks"


## 2. One sentence, two roles for <i>fish</i>

Consider:

> They can fish.  
> The fish swim.

The spelling of <i>fish</i> is unchanged, but its tag depends on its context. In the first sentence it is a verb; in the second it is a noun.

**Predict before running the next cell**

- Which Penn Treebank tags would you assign?
- What evidence comes from the word itself?
- What evidence comes from nearby tags or words?

In [3]:
toy_tagged = [
    [("They", "PRP"), ("can", "MD"), ("fish", "VB")],
    [("We", "PRP"), ("can", "MD"), ("fish", "VB")],
    [("People", "NNS"), ("can", "MD"), ("fish", "VB")],
    [("The", "DT"), ("fish", "NN"), ("swim", "VBP")],
    [("A", "DT"), ("fish", "NN"), ("escaped", "VBD")],
]

for sentence in toy_tagged:
    print(" ".join(f"{word}/{tag}" for word, tag in sentence))

They/PRP can/MD fish/VB
We/PRP can/MD fish/VB
People/NNS can/MD fish/VB
The/DT fish/NN swim/VBP
A/DT fish/NN escaped/VBD


## 3. HMM tagging: tags are hidden states

For an HMM tagger:

- the **observations** are the words we can see;
- the **states** are the PoS tags we want to infer;
- a **transition probability** \(P(t_i \mid t_{i-1})\) describes how likely one tag is after another;
- an **emission probability** \(P(w_i \mid t_i)\) describes how likely a tag is to emit a word.

For words \(w_1,\ldots,w_n\) and tags \(t_1,\ldots,t_n\), a first-order HMM assigns a joint probability of the form

\[
P(t_1)P(w_1\mid t_1)
\prod_{i=2}^{n} P(t_i\mid t_{i-1})P(w_i\mid t_i).
\]

The Viterbi algorithm efficiently finds the most probable tag sequence for the observed words.

This is a **sequence decision**: the tagger compares complete paths rather than choosing each tag independently.

In [4]:
def lidstone_estimator(freqdist, bins):
    # A small amount of smoothing prevents zero probabilities in this toy model.
    return LidstoneProbDist(freqdist, gamma=0.1, bins=bins + 1)


toy_hmm = HiddenMarkovModelTrainer().train_supervised(
    toy_tagged,
    estimator=lidstone_estimator,
)

for words in [
    ["They", "can", "fish"],
    ["The", "fish", "escaped"],
]:
    print(toy_hmm.tag(words))

[('They', 'PRP'), ('can', 'MD'), ('fish', 'VB')]
[('The', 'DT'), ('fish', 'NN'), ('escaped', 'VBD')]


### Inspect two candidate paths

For <i>They can fish</i>, compare:

- <code>PRP → MD → VB</code>, the intended analysis;
- <code>PRP → VB → NN</code>, a competing analysis.

The values below are joint probabilities. Their absolute size is not important here; their ordering is.

In [5]:
candidate_paths = {
    "PRP → MD → VB": [("They", "PRP"), ("can", "MD"), ("fish", "VB")],
    "PRP → VB → NN": [("They", "PRP"), ("can", "VB"), ("fish", "NN")],
}

rows = []
for path_name, tagged_sequence in candidate_paths.items():
    rows.append(
        {
            "Candidate tag path": path_name,
            "Joint probability": toy_hmm.probability(tagged_sequence),
        }
    )

display(pd.DataFrame(rows).sort_values("Joint probability", ascending=False))

,Candidate tag path,Joint probability
0,PRP → MD → VB,0.045119
1,PRP → VB → NN,0.000009


**Observe and explain**

1. Which path receives the higher probability?
2. Which transitions support it?
3. Which emissions support it?
4. Why is the word <i>fish</i> alone insufficient?

Do not reduce the explanation to “the model knows that <i>fish</i> is a verb”. The model combines evidence about words and tag sequences.

## 4. Brill tagging: correct a simple baseline

A Brill tagger follows a different strategy:

1. tag the training text with a simple baseline;
2. compare the baseline tags with the gold-standard tags;
3. find a contextual transformation that corrects many errors while introducing few new ones;
4. apply the best rule;
5. repeat.

The result is an ordered list of readable correction rules.

We deliberately create a weak baseline below. It memorises the most frequent tag for each known word. Unknown words receive <code>NN</code>.

In [6]:
toy_baseline = UnigramTagger(
    toy_tagged,
    backoff=DefaultTagger("NN"),
)

ambiguous_sentence = "The fish escaped".split()
print("Baseline:", toy_baseline.tag(ambiguous_sentence))

Baseline: [('The', 'DT'), ('fish', 'VB'), ('escaped', 'VBD')]


The training corpus contains <i>fish</i> three times as <code>VB</code> and twice as <code>NN</code>, so a word-only baseline tags every occurrence as <code>VB</code>. It needs contextual evidence to repair <i>The fish</i>.

In [7]:
toy_templates = [
    brill.Template(brill.Pos([-1])),
    brill.Template(brill.Pos([1])),
    brill.Template(brill.Word([-1])),
    brill.Template(brill.Word([1])),
]

toy_brill_trainer = BrillTaggerTrainer(
    toy_baseline,
    toy_templates,
    trace=0,
)
toy_brill = toy_brill_trainer.train(
    toy_tagged,
    max_rules=10,
    min_score=1,
)

print("Learned rule(s):")
for rule in toy_brill.rules():
    print(" ", rule)

print("\nAfter Brill correction:", toy_brill.tag(ambiguous_sentence))

Learned rule(s):
  VB->NN if Pos:DT@[-1]

After Brill correction: [('The', 'DT'), ('fish', 'NN'), ('escaped', 'VBD')]


**Observe and explain**

1. Translate the learned rule into plain English.
2. Which baseline error does it correct?
3. Would it correct every possible use of <i>fish</i>? Why not?

At this point the two taggers have seen the same labelled examples, but place their generalisations in different forms:

| HMM | Brill |
|---|---|
| probabilities over transitions and emissions | an initial tagger plus ordered correction rules |
| compares complete tag sequences | repeatedly transforms a baseline sequence |
| ambiguity resolved by relative path probability | ambiguity resolved when a learned rule's conditions apply |

## 5. Scale up: held-out WSJ text

We now use NLTK's Penn Treebank sample. To keep training and evaluation separate, whole WSJ files—not individual sentences—are divided into:

- **training files**, used to estimate models and learn rules;
- **test files**, used only for evaluation.

The test labels remain visible to us because they are needed to measure accuracy. “Held out” means excluded from fitting; it does not mean hidden from the student.

Penn Treebank traces tagged <code>-NONE-</code> are removed because they are not surface tokens that a tagger would receive in ordinary text.

In [8]:
def clean_tagged_sents(sentences):
    cleaned = []
    for sentence in sentences:
        surface = [(word, tag) for word, tag in sentence if tag != "-NONE-"]
        if surface:
            cleaned.append(surface)
    return cleaned


fileids = sorted(treebank.fileids())
train_fileids = fileids[:150]
test_fileids = fileids[150:]

wsj_train = clean_tagged_sents(treebank.tagged_sents(train_fileids))
wsj_test = clean_tagged_sents(treebank.tagged_sents(test_fileids))

print(f"Training: {len(train_fileids)} files, {len(wsj_train):,} sentences, "
      f"{sum(map(len, wsj_train)):,} tokens")
print(f"Test:     {len(test_fileids)} files, {len(wsj_test):,} sentences, "
      f"{sum(map(len, wsj_test)):,} tokens")

print("\nExample training sentence:")
print(" ".join(f"{word}/{tag}" for word, tag in wsj_train[0][:25]))

Training: 150 files, 3,262 sentences, 78,539 tokens
Test:     49 files, 652 sentences, 15,545 tokens

Example training sentence:
Pierre/NNP Vinken/NNP ,/, 61/CD years/NNS old/JJ ,/, will/MD join/VB the/DT board/NN as/IN a/DT nonexecutive/JJ director/NN Nov./NNP 29/CD ./.


### 5.1 Train the HMM

Real test text contains words absent from training. An unsmoothed lexical model cannot estimate an emission probability for them.

Here, words occurring only once in the WSJ training material are replaced by <code>&lt;UNK&gt;</code>. The HMM therefore learns how different tags emit an unknown-word symbol. At tagging time, any word outside its retained vocabulary is mapped to that symbol. We also apply mild Lidstone smoothing to the learned distributions.

This is a modest teaching solution to the unknown-word problem, not the only possible solution.

In [9]:
training_word_counts = Counter(
    word for sentence in wsj_train for word, _ in sentence
)
hmm_vocabulary = {
    word for word, count in training_word_counts.items() if count >= 2
}


def hmm_word(word):
    return word if word in hmm_vocabulary else "<UNK>"


hmm_train = [
    [(hmm_word(word), tag) for word, tag in sentence]
    for sentence in wsj_train
]

start = time.perf_counter()
wsj_hmm = HiddenMarkovModelTrainer().train_supervised(
    hmm_train,
    estimator=lidstone_estimator,
)
hmm_training_seconds = time.perf_counter() - start

print(f"Retained HMM vocabulary: {len(hmm_vocabulary):,} word types")
print(f"HMM training time: {hmm_training_seconds:.2f} seconds")

Retained HMM vocabulary: 5,145 word types
HMM training time: 0.10 seconds


### 5.2 Train the Brill system

The Brill tagger needs a baseline to correct. Ours has two layers:

1. a unigram tagger memorises the most frequent training tag for known words;
2. simple suffix and word-shape patterns provide guesses for unknown words.

The Brill trainer then learns contextual corrections to that baseline. Unlike the HMM's <code>&lt;UNK&gt;</code> treatment, the baseline can exploit the spelling of an unseen word.

In [10]:
unknown_word_patterns = [
    (r"^-?[0-9]+(?:\.[0-9]+)?$", "CD"),
    (r".*ing$", "VBG"),
    (r".*ed$", "VBD"),
    (r".*ly$", "RB"),
    (r".*ous$", "JJ"),
    (r".*able$", "JJ"),
    (r".*s$", "NNS"),
    (r".*", "NN"),
]

regexp_backoff = RegexpTagger(unknown_word_patterns)
wsj_baseline = UnigramTagger(wsj_train, backoff=regexp_backoff)

wsj_templates = [
    brill.Template(brill.Pos([-1])),
    brill.Template(brill.Pos([1])),
    brill.Template(brill.Pos([-2])),
    brill.Template(brill.Pos([2])),
    brill.Template(brill.Word([-1])),
    brill.Template(brill.Word([1])),
    brill.Template(brill.Pos([-1]), brill.Word([0])),
    brill.Template(brill.Pos([1]), brill.Word([0])),
]

start = time.perf_counter()
wsj_brill = BrillTaggerTrainer(
    wsj_baseline,
    wsj_templates,
    trace=0,
).train(
    wsj_train,
    max_rules=75,
    min_score=3,
)
brill_training_seconds = time.perf_counter() - start

print(f"Brill training time: {brill_training_seconds:.2f} seconds")
print(f"Rules learned: {len(wsj_brill.rules())}")

Brill training time: 1.63 seconds
Rules learned: 75


### Inspect the learned Brill rules

The score shown in a rule is its net improvement on the training material when it was learned: corrections minus newly introduced errors.

In [11]:
for number, rule in enumerate(wsj_brill.rules()[:12], start=1):
    print(f"{number:>2}. {rule}")

 1. NN->VB if Pos:TO@[-1]
 2. VBP->VB if Pos:MD@[-1]
 3. VBP->VB if Word:to@[-1]
 4. POS->VBZ if Pos:PRP@[-1]
 5. NN->VB if Pos:MD@[-1]
 6. VB->NN if Pos:DT@[-1]
 7. IN->WDT if Pos:VBZ@[1] & Word:that@[0]
 8. VBN->VBD if Pos:PRP@[-1]
 9. VBD->VBN if Pos:VBP@[-1]
10. IN->WDT if Pos:NNS@[-1] & Word:that@[0]
11. VBD->VBN if Pos:VBZ@[-1]
12. VB->VBP if Pos:NNS@[-1]


**Choose two rules and explain them**

- What tag is changed?
- Under what contextual condition?
- Why might the correction often be useful in WSJ prose?
- Can you invent a sentence in which it would be wrong?

## 6. Evaluate on held-out WSJ sentences

Overall token accuracy is:

\[
\mathrm{accuracy}
=
\frac{\text{number of correctly tagged tokens}}
{\text{number of test tokens}}.
\]

Accuracy is useful, but it can conceal systematic confusions. We shall calculate it first, then inspect errors.

In [12]:
def predict_sentence(tagger_name, sentence):
    words = [word for word, _ in sentence]
    if tagger_name == "HMM":
        model_input = [hmm_word(word) for word in words]
        predicted_tags = [tag for _, tag in wsj_hmm.tag(model_input)]
    elif tagger_name == "Baseline":
        predicted_tags = [tag for _, tag in wsj_baseline.tag(words)]
    elif tagger_name == "Brill":
        predicted_tags = [tag for _, tag in wsj_brill.tag(words)]
    else:
        raise ValueError(f"Unknown tagger: {tagger_name}")
    return words, predicted_tags


def evaluate_tagger(tagger_name, tagged_sentences):
    gold_tags = []
    predicted_tags = []
    records = []

    for sentence_number, sentence in enumerate(tagged_sentences):
        words, sentence_predictions = predict_sentence(tagger_name, sentence)
        for token_number, ((word, gold), predicted) in enumerate(
            zip(sentence, sentence_predictions)
        ):
            gold_tags.append(gold)
            predicted_tags.append(predicted)
            records.append(
                {
                    "sentence": sentence_number,
                    "token": token_number,
                    "word": word,
                    "gold": gold,
                    "predicted": predicted,
                }
            )

    correct = sum(gold == predicted for gold, predicted in zip(gold_tags, predicted_tags))
    return correct / len(gold_tags), pd.DataFrame(records)


evaluation_rows = []
evaluation_records = {}

for tagger_name in ["Baseline", "HMM", "Brill"]:
    start = time.perf_counter()
    accuracy, records = evaluate_tagger(tagger_name, wsj_test)
    elapsed = time.perf_counter() - start
    evaluation_records[tagger_name] = records
    evaluation_rows.append(
        {
            "Tagger": tagger_name,
            "Held-out token accuracy": accuracy,
            "Evaluation time (s)": elapsed,
        }
    )

evaluation_table = pd.DataFrame(evaluation_rows)
evaluation_for_display = evaluation_table.copy()
evaluation_for_display["Held-out token accuracy"] = (
    evaluation_for_display["Held-out token accuracy"].map(lambda value: f"{value:.3%}")
)
evaluation_for_display["Evaluation time (s)"] = (
    evaluation_for_display["Evaluation time (s)"].map(lambda value: f"{value:.2f}")
)
display(evaluation_for_display)

,Tagger,Held-out token accuracy,Evaluation time (s)
0,Baseline,89.739%,0.09
1,HMM,90.466%,4.55
2,Brill,91.792%,0.13


**Observe and explain**

1. How much does Brill improve on its baseline?
2. How does the HMM compare?
3. Does one accuracy figure establish that one approach is universally better? Consider the training choices, templates, unknown-word handling and domain.
4. Which comparison isolates the contribution of the learned Brill transformations most directly?

### 6.1 Which tag confusions are most frequent?

The next table ignores correct predictions and counts pairs of the form:

> gold tag → predicted tag

In [13]:
def top_confusions(records, number=12):
    errors = records[records["gold"] != records["predicted"]]
    counts = (
        errors.groupby(["gold", "predicted"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(number)
    )
    return counts


for tagger_name in ["HMM", "Brill"]:
    print(f"\n{tagger_name}: most frequent confusions")
    display(top_confusions(evaluation_records[tagger_name]))


HMM: most frequent confusions


,gold,predicted,count
55,NN,JJ,106
96,NNS,NN,99
76,NNP,JJ,95
57,NN,NNP,94
77,NNP,NN,59
2,CD,JJ,57
58,NN,NNS,52
97,NNS,NNP,46
30,JJ,NNP,44
168,VBN,JJ,33



Brill: most frequent confusions


,gold,predicted,count
39,NNP,NN,412
14,JJ,NN,133
74,VBD,VBN,63
41,NNP,NNS,53
80,VBN,VBD,49
19,JJ,VBD,33
2,CD,NN,33
32,NN,NNS,32
76,VBG,NN,25
29,NN,JJ,23


**Investigate**

Choose one frequent confusion from each tagger.

1. Find examples in the records below.
2. Look at the surrounding words.
3. Is the error understandable from the model's available evidence?
4. Would a larger context necessarily solve it?

In [14]:
def show_error_examples(tagger_name, gold, predicted, number=8):
    records = evaluation_records[tagger_name]
    matches = records[
        (records["gold"] == gold) & (records["predicted"] == predicted)
    ].head(number)

    rows = []
    for row in matches.itertuples():
        sentence = wsj_test[row.sentence]
        left = max(0, row.token - 4)
        right = min(len(sentence), row.token + 5)
        context = " ".join(word for word, _ in sentence[left:right])
        rows.append(
            {
                "word": row.word,
                "gold": row.gold,
                "predicted": row.predicted,
                "context": context,
            }
        )
    return pd.DataFrame(rows)


# Change these tags after inspecting the confusion tables.
display(show_error_examples("HMM", gold="NN", predicted="JJ"))
display(show_error_examples("Brill", gold="NN", predicted="JJ"))

,word,gold,predicted,context
0,printing-press,NN,JJ,"supply , electronics and printing-press concer..."
1,passenger-car,NN,JJ,in the heavy-truck and passenger-car markets .
2,Aerospace,NN,JJ,Aerospace earnings sagged 37 %
3,B-1B,NN,JJ,largely due to lower B-1B program profit ; the
4,C-130,NN,JJ,"program -- probably the C-130 gunship , accord..."
5,net,NN,JJ,", the company posted net of $ 734.9 million"
6,color,NN,JJ,hard disk and a color monitor .
7,automobile,NN,JJ,to produce alcohol for automobile fuel .


,word,gold,predicted,context
0,buy-back,NN,JJ,"Mr. Ackerman said the buy-back , and the above..."
1,net,NN,JJ,", the company posted net of $ 734.9 million"
2,executive,NN,JJ,'s president and chief executive officer .
3,executive,NN,JJ,on Reuters 's eight-person executive committee...
4,depositary,NN,JJ,"over-the-counter market , American depositary ..."
5,senior,NN,JJ,The senior of the three executives
6,average,NN,JJ,points in the industrial average .
7,executive,NN,JJ,as president and chief executive officer spark...


### 6.2 What happens to unseen words?

An unseen word is a word type absent from the WSJ training split. This is a stricter definition than the HMM's rare-word threshold: some words seen once in training are deliberately represented as <code>&lt;UNK&gt;</code> by the HMM.

In [15]:
training_vocabulary_all = set(training_word_counts)


def unseen_word_accuracy(records):
    unseen = records[~records["word"].isin(training_vocabulary_all)]
    accuracy = (unseen["gold"] == unseen["predicted"]).mean()
    return len(unseen), accuracy


unseen_rows = []
for tagger_name in ["Baseline", "HMM", "Brill"]:
    count, accuracy = unseen_word_accuracy(evaluation_records[tagger_name])
    unseen_rows.append(
        {
            "Tagger": tagger_name,
            "Unseen test tokens": count,
            "Accuracy on unseen tokens": accuracy,
        }
    )

unseen_table = pd.DataFrame(unseen_rows)
unseen_table["Accuracy on unseen tokens"] = unseen_table[
    "Accuracy on unseen tokens"
].map(lambda value: f"{value:.3%}")
display(unseen_table)

,Tagger,Unseen test tokens,Accuracy on unseen tokens
0,Baseline,1533,43.575%
1,HMM,1533,54.860%
2,Brill,1533,48.337%


**Explain**

- What evidence can the HMM use for an unseen word after it has been mapped to <code>&lt;UNK&gt;</code>?
- What evidence can the Brill system's baseline use?
- Why can the ordered Brill rules still affect an unseen word after the baseline has assigned a tag?

## 7. From a test corpus to a complete book

We now apply both trained taggers to Lewis Carroll's *Alice's Adventures in Wonderland*.

This is **domain transfer**:

- training: financial newswire from the WSJ;
- application: nineteenth-century literary prose.

The book has no Penn Treebank gold tags in this notebook. We can inspect outputs, disagreements, coverage and running time, but we **cannot calculate accuracy** merely by looking at how plausible the tags seem.

In [16]:
alice_sentences_with_front_matter = [
    list(sentence)
    for sentence in gutenberg.sents("carroll-alice.txt")
    if sentence
]
# Omit the Gutenberg title line, chapter number and chapter title.
alice_sentences = alice_sentences_with_front_matter[3:]
alice_words = [word for sentence in alice_sentences for word in sentence]

print(f"Book sentences: {len(alice_sentences):,}")
print(f"Book tokens:    {len(alice_words):,}")
print(f"Distinct words: {len(set(alice_words)):,}")

alice_unseen = sum(word not in training_vocabulary_all for word in alice_words)
print(f"Tokens unseen in WSJ training: {alice_unseen:,} "
      f"({alice_unseen / len(alice_words):.1%})")

Book sentences: 1,700
Book tokens:    34,093
Distinct words: 3,010
Tokens unseen in WSJ training: 6,618 (19.4%)


### Inspect a few sentences first

The HMM is shown the transformed <code>&lt;UNK&gt;</code> sequence internally, but we display each predicted tag beside the original book word.

In [17]:
def tag_words_for_display(words):
    hmm_input = [hmm_word(word) for word in words]
    hmm_tags = [tag for _, tag in wsj_hmm.tag(hmm_input)]
    brill_tags = [tag for _, tag in wsj_brill.tag(words)]
    return list(zip(words, hmm_tags, brill_tags))


for sentence in alice_sentences[5:8]:
    print("\nTEXT:", " ".join(sentence))
    display(
        pd.DataFrame(
            tag_words_for_display(sentence),
            columns=["Word", "HMM tag", "Brill tag"],
        )
    )


TEXT: ( when she thought it over afterwards , it occurred to her that she ought to have wondered at this , but at the time it all seemed quite natural ); but when the Rabbit actually TOOK A WATCH OUT OF ITS WAISTCOAT - POCKET , and looked at it , and then hurried on , Alice started to her feet , for it flashed across her mind that she had never before seen a rabbit with either a waistcoat - pocket , or a watch to take out of it , and burning with curiosity , she ran across the field after it , and fortunately was just in time to see it pop down a large rabbit - hole under the hedge .


,Word,HMM tag,Brill tag
0,(,NNS,NN
1,when,WRB,WRB
2,she,PRP,PRP
3,thought,VBD,VBD
4,it,PRP,PRP
...,...,...,...
120,hole,CD,NN
121,under,IN,IN
122,the,DT,DT
123,hedge,NN,NN



TEXT: In another moment down went Alice after it , never once considering how in the world she was to get out again .


,Word,HMM tag,Brill tag
0,In,IN,IN
1,another,DT,DT
2,moment,NN,NN
3,down,RB,RB
4,went,VBD,VBD
5,Alice,VBN,NN
6,after,IN,IN
7,it,PRP,PRP
8,",",",",","
9,never,RB,RB



TEXT: The rabbit - hole went straight on like a tunnel for some way , and then dipped suddenly down , so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well .


,Word,HMM tag,Brill tag
0,The,DT,DT
1,rabbit,NN,NN
2,-,:,:
3,hole,NNP,NN
4,went,VBD,VBD
5,straight,RB,JJ
6,on,IN,IN
7,like,IN,IN
8,a,DT,DT
9,tunnel,NN,NN


### Apply both taggers to the full book

We retain only compact aggregate information rather than printing tens of thousands of tagged tokens.

In [18]:
start = time.perf_counter()
alice_hmm_tags = []
for sentence in alice_sentences:
    hmm_input = [hmm_word(word) for word in sentence]
    alice_hmm_tags.append([tag for _, tag in wsj_hmm.tag(hmm_input)])
hmm_book_seconds = time.perf_counter() - start

start = time.perf_counter()
alice_brill_tags = [
    [tag for _, tag in wsj_brill.tag(sentence)]
    for sentence in alice_sentences
]
brill_book_seconds = time.perf_counter() - start

book_disagreements = sum(
    hmm_tag != brill_tag
    for hmm_sentence, brill_sentence in zip(alice_hmm_tags, alice_brill_tags)
    for hmm_tag, brill_tag in zip(hmm_sentence, brill_sentence)
)

book_summary = pd.DataFrame(
    [
        ("HMM", hmm_book_seconds),
        ("Brill", brill_book_seconds),
    ],
    columns=["Tagger", "Time for full book (s)"],
)

book_summary["Time for full book (s)"] = book_summary[
    "Time for full book (s)"
].map(lambda value: f"{value:.2f}")
display(book_summary)
print(
    f"Taggers disagree on {book_disagreements:,} of {len(alice_words):,} tokens "
    f"({book_disagreements / len(alice_words):.1%})."
)

,Tagger,Time for full book (s)
0,HMM,7.63
1,Brill,0.23


Taggers disagree on 8,899 of 34,093 tokens (26.1%).


### Inspect disagreements

A disagreement is not automatically an error. Without gold labels, it tells us where the models make different decisions and gives us a useful place to inspect.

In [19]:
disagreement_rows = []

for sentence_number, (sentence, hmm_tags, brill_tags) in enumerate(
    zip(alice_sentences, alice_hmm_tags, alice_brill_tags)
):
    for token_number, (word, hmm_tag, brill_tag) in enumerate(
        zip(sentence, hmm_tags, brill_tags)
    ):
        if hmm_tag != brill_tag:
            left = max(0, token_number - 4)
            right = min(len(sentence), token_number + 5)
            disagreement_rows.append(
                {
                    "word": word,
                    "HMM": hmm_tag,
                    "Brill": brill_tag,
                    "context": " ".join(sentence[left:right]),
                }
            )

alice_disagreements = pd.DataFrame(disagreement_rows)
display(alice_disagreements.head(25))

,word,HMM,Brill,context
0,Alice,NNP,NN,Alice was beginning to get
1,tired,VBN,JJ,beginning to get very tired of sitting by her
2,sitting,NN,VBG,get very tired of sitting by her sister on
3,twice,VBG,RB,do : once or twice she had peeped into
4,pictures,NN,NNS,but it had no pictures or conversations in it
5,','',POS,"conversations in it , ' and what is the"
6,",'",NNS,NN,"use of a book ,' thought Alice ' without"
7,Alice,NNP,NN,"a book ,' thought Alice ' without pictures or"
8,pictures,NNP,NNS,thought Alice ' without pictures or conversati...
9,conversation,NNP,NN,' without pictures or conversation ?'


**Required discussion**

Choose five disagreements and make a reasoned judgement about each.

1. Which tag seems more plausible in context?
2. What evidence might have led the HMM to its decision?
3. Which baseline guess or learned Brill rule might have led to the Brill decision?
4. Is the word unseen in WSJ training?
5. Is the difficulty caused by genuine ambiguity, genre change, tokenisation, old-fashioned usage, or something else?

Record uncertainty where appropriate. Manual inspection of five examples is qualitative analysis, not a new accuracy estimate.

## 8. Synthesis — where is the generalisation?

Complete these prompts in your notes.

1. **PoS tagging is more than dictionary lookup because ...**
2. **The HMM generalises by ...**
3. **The Brill tagger generalises by ...**
4. **Held-out WSJ accuracy tells us ... but does not tell us ...**
5. **Applying the taggers to *Alice* demonstrates scaling and domain transfer, but it does not evaluate accuracy because ...**
6. **One useful thing we learned from inspecting errors that an accuracy score concealed was ...**

The two taggers use the same broad source of evidence—labelled word/tag sequences—but encode that evidence differently:

> **HMM: probabilistic sequence model**  
> **Brill: baseline plus explicit contextual transformations**

Modern taggers may use learned representations and much wider context, but the central questions remain: what evidence is available, how is context represented, how does the model generalise, and how do we know whether its predictions are good?

## Optional further investigation

If time permits:

- increase or decrease the number of Brill rules and compare training time, WSJ test accuracy and the top learned rules;
- change the HMM rare-word threshold and examine unseen-word accuracy;
- compare lower-cased and original text, stating what information lower-casing removes;
- find a sentence on which both taggers agree but appear to be wrong;
- repeat the application stage with another public-domain Gutenberg text.

Change one thing at a time:

> **Predict → run → observe → explain.**